<div style="border-top: 4px solid #2563eb; padding: 48px 48px 36px; background: #f8fafc; margin-bottom: 0;">
  <h1 style="color: #0f172a; font-family: 'Inter', 'Segoe UI', sans-serif; font-size: 2.2em; font-weight: 700; margin: 0 0 10px 0; letter-spacing: -0.5px;">COVID-19 Global Impact Analysis</h1>
  <p style="color: #475569; font-size: 1.05em; margin: 0 0 28px 0; font-family: 'Inter', sans-serif; line-height: 1.6;">An interactive analysis of cases, deaths, wave patterns, and vaccination coverage using WHO and OWID data.</p>
  <div style="display: flex; gap: 16px; flex-wrap: wrap; border-top: 1px solid #e2e8f0; padding-top: 20px;">
    <span style="color: #6b5d48; font-size: 0.82em; font-family: 'Inter', sans-serif;">Parthiv Patel &mdash; parthiv.patel@iitgn.ac.in</span>
    <span style="color: #c4bba8;">|</span>
    <span style="color: #6b5d48; font-size: 0.82em; font-family: 'Inter', sans-serif;">Aditya Borate &mdash; aditya.borate@iitgn.ac.in</span>
    <span style="color: #c4bba8;">|</span>
    <span style="color: #6b5d48; font-size: 0.82em; font-family: 'Inter', sans-serif;">Srajan Dehariya &mdash; srajan.dehariya@iitgn.ac.in</span>
    <span style="color: #c4bba8;">|</span>
    <span style="color: #6b5d48; font-size: 0.82em; font-family: 'Inter', sans-serif;">Rudra Pratap Singh (23110281) &mdash; rudra.pratap@iitgn.ac.in</span>
    <span style="color: #c4bba8;">|</span>
    <span style="color: #6b5d48; font-size: 0.82em; font-family: 'Inter', sans-serif;">CS328</span>
  </div>
</div>

<div style="background: #f1f5f9; border-left: 3px solid #2563eb; padding: 20px 28px; margin: 0 0 8px 0;">
  <h2 style="color: #1e40af; font-family: 'Inter', sans-serif; font-size: 1em; font-weight: 600; margin: 0 0 8px; text-transform: uppercase; letter-spacing: 0.06em;">Introduction</h2>
  <p style="color: #374151; font-family: 'Inter', sans-serif; line-height: 1.8; margin: 0; font-size: 0.97em;">
    The COVID-19 pandemic, originating in December 2019, became the defining global crisis of the 21st century. It reshaped healthcare, economies, and daily life across every continent. This analysis uses <strong>WHO and OWID datasets</strong> to reveal the pandemic's trajectory &mdash; from regional spread patterns and fatality rates to country-level wave dynamics and global case geography.
  </p>
</div>

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly
import warnings
warnings.filterwarnings('ignore')
plotly.offline.init_notebook_mode()

# ── Light theme defaults ──────────────────────────────────────────────────────
LIGHT = dict(
    paper_bgcolor='#faf8f4',
    plot_bgcolor='#f5f1e8',
    font=dict(family='Inter, Segoe UI, sans-serif', color='#3d3120', size=12),
    title_font=dict(size=17, color='#2d2416', family='Inter, Segoe UI, sans-serif'),
    legend=dict(bgcolor='rgba(250,248,244,0.92)', bordercolor='#d6cfc0', borderwidth=1,
                font=dict(color='#3d3120', size=11)),
    margin=dict(t=70, b=55, l=65, r=30),
)

# Refined palette: accessible, print-friendly
PALETTE = ['#2563eb', '#16a34a', '#dc2626', '#d97706', '#7c3aed', '#0891b2']
PALETTE8 = ['#2563eb','#16a34a','#dc2626','#d97706','#7c3aed','#0891b2','#c2410c','#4f46e5']

def apply_light_axes(fig, xgrid=True, ygrid=True):
    """Apply consistent light-mode axis styling."""
    fig.update_xaxes(
        showgrid=xgrid, gridcolor='#e5e7eb', gridwidth=1,
        showline=True, linecolor='#d1d5db', linewidth=1,
        zeroline=False, tickfont=dict(color='#6b7280', size=11)
    )
    fig.update_yaxes(
        showgrid=ygrid, gridcolor='#e5e7eb', gridwidth=1,
        showline=True, linecolor='#d1d5db', linewidth=1,
        zeroline=False, tickfont=dict(color='#6b7280', size=11)
    )
    return fig

print('Environment ready | plotly', plotly.__version__)

In [ ]:
# ── Load WHO regional table ──────────────────────────────────────────────────
df_table = pd.read_csv('./data/WHO-COVID-19-global-table-data.csv')
df_table = df_table.drop(df_table[df_table.iloc[:, 3] == 0].index)

col_drop = [
    'Cases - cumulative total per 100000 population',
    'Cases - newly reported in last 7 days per 100000 population',
    'Deaths - cumulative total per 100000 population',
    'Deaths - newly reported in last 7 days per 100000 population'
]
covid_df = df_table.drop(col_drop, axis=1)

regions = list(set(df_table['WHO Region']) - {'abs', 'Other'})
region_map = {r: [] for r in regions}
for _, row in df_table.iterrows():
    if row['WHO Region'] in region_map:
        region_map[row['WHO Region']].append(row['Name'])

print(f'{len(df_table)} countries loaded | {len(regions)} WHO regions')
covid_df.head(3)

<div style="border-top: 2px solid #e2e8f0; padding-top: 32px; margin: 36px 0 20px;">
  <span style="background: #dbeafe; color: #1d4ed8; font-size: 0.75em; font-weight: 600; font-family: 'Inter', sans-serif; text-transform: uppercase; letter-spacing: 0.08em; padding: 4px 10px; border-radius: 4px;">Section 1</span>
  <h2 id="section-1" style="color: #0f172a; font-family: 'Inter', sans-serif; font-size: 1.35em; font-weight: 700; margin: 10px 0 4px;">Regional Case Distribution</h2>
  <p style="color: #64748b; font-family: 'Inter', sans-serif; font-size: 0.95em; margin: 0;">How the pandemic distributed across WHO's six geographic regions.</p>
</div>

In [ ]:
# ── Aggregate per region ─────────────────────────────────────────────────────
agg = {r: {'cases_total': 0, 'cases_7d': 0, 'cases_24h': 0,
            'deaths_total': 0, 'deaths_7d': 0, 'deaths_24h': 0} for r in regions}

for _, row in covid_df.iterrows():
    reg = row['WHO Region']
    if reg in agg:
        agg[reg]['cases_total']  += row.get('Cases - cumulative total', 0)
        agg[reg]['cases_7d']     += row.get('Cases - newly reported in last 7 days', 0)
        agg[reg]['cases_24h']    += row.get('Cases - newly reported in last 24 hours', 0)
        agg[reg]['deaths_total'] += row.get('Deaths - cumulative total', 0)
        agg[reg]['deaths_7d']    += row.get('Deaths - newly reported in last 7 days', 0)
        agg[reg]['deaths_24h']   += row.get('Deaths - newly reported in last 24 hours', 0)

agg_df = pd.DataFrame(agg).T.reset_index().rename(columns={'index': 'Region'})
agg_df = agg_df.sort_values('cases_total', ascending=False)
agg_df['n_countries'] = [len(region_map.get(r, [])) for r in agg_df['Region']]

# ── Grouped bar: Cumulative vs 7-day (SAME INFO as before, different grouping style) ─
fig = go.Figure()
fig.add_trace(go.Bar(
    name='Cumulative Cases', x=agg_df['Region'], y=agg_df['cases_total'],
    marker_color='#2563eb', marker_line_color='#1d4ed8', marker_line_width=0.5,
    hovertemplate='<b>%{x}</b><br>Total: %{y:,.0f}<extra></extra>'
))
fig.add_trace(go.Bar(
    name='Last 7 Days', x=agg_df['Region'], y=agg_df['cases_7d'],
    marker_color='#93c5fd', marker_line_color='#60a5fa', marker_line_width=0.5,
    hovertemplate='<b>%{x}</b><br>7-day new: %{y:,.0f}<extra></extra>'
))
fig.update_layout(
    **LIGHT,
    title='COVID-19 Cases by WHO Region — Cumulative vs Recent 7-Day New Cases',
    barmode='group',
    yaxis_title='Number of Cases',
    height=420,
)
apply_light_axes(fig)
fig.show()

In [ ]:
# ── Stacked horizontal bar: Cases + Deaths share (same data as old donut, different form) ─
# Previously shown as two donut charts; now a stacked proportional bar for easier comparison
total_cases = agg_df['cases_total'].sum()
total_deaths = agg_df['deaths_total'].sum()
agg_sorted = agg_df.sort_values('cases_total')

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Share of Cumulative Cases (%)', 'Share of Cumulative Deaths (%)'])

for i, (col, total) in enumerate([('cases_total', total_cases),
                                    ('deaths_total', total_deaths)], 1):
    pct = 100 * agg_sorted[col] / total
    for j, (region, p) in enumerate(zip(agg_sorted['Region'], pct)):
        fig.add_trace(go.Bar(
            name=region, x=[p], y=['WHO Regions'],
            orientation='h',
            marker_color=PALETTE[j % len(PALETTE)],
            text=f'{region}<br>{p:.1f}%',
            textposition='inside',
            insidetextanchor='middle',
            hovertemplate=f'<b>{region}</b><br>{col.replace("_"," ").title()}: %{{x:.1f}}%<extra></extra>',
            showlegend=(i == 1),
        ), row=1, col=i)

fig.update_layout(
    **LIGHT,
    barmode='stack',
    title='Regional Share of Total COVID-19 Burden',
    height=280,
    showlegend=True,
)
apply_light_axes(fig, xgrid=False, ygrid=False)
fig.update_xaxes(range=[0, 100], ticksuffix='%')
fig.show()

<div style="background: #eff6ff; border: 1px solid #bfdbfe; border-radius: 6px; padding: 16px 22px; margin: 8px 0 24px;">
  <p style="color: #1e3a5f; font-family: 'Inter', sans-serif; line-height: 1.75; margin: 0; font-size: 0.94em;">
    <strong>Key insight:</strong> Despite having fewer member countries, the European and American regions account for the largest share of cumulative cases. Africa &mdash; home to the most nations &mdash; shows a markedly smaller share, partly due to reporting gaps and differing age demographics.
  </p>
</div>

<div style="border-top: 2px solid #e2e8f0; padding-top: 32px; margin: 36px 0 20px;">
  <span style="background: #f0fdf4; color: #15803d; font-size: 0.75em; font-weight: 600; font-family: 'Inter', sans-serif; text-transform: uppercase; letter-spacing: 0.08em; padding: 4px 10px; border-radius: 4px;">Section 2</span>
  <h2 id="section-2" style="color: #0f172a; font-family: 'Inter', sans-serif; font-size: 1.35em; font-weight: 700; margin: 10px 0 4px;">Comparative Case Study: The Americas vs. India</h2>
  <p style="color: #64748b; font-family: 'Inter', sans-serif; font-size: 0.95em; margin: 0;">Comparing absolute scale and fatality consequences between two distinct global epicentres.</p>
</div>

In [ ]:
# ── Case-fatality rates ──────────────────────────────────────────────────────
agg_df['fatality_rate_total'] = np.where(
    agg_df['cases_total'] > 0,
    100 * agg_df['deaths_total'] / agg_df['cases_total'], 0
)
agg_df['fatality_rate_24h'] = np.where(
    agg_df['cases_24h'] > 0,
    100 * agg_df['deaths_24h'] / agg_df['cases_24h'], 0
)

# Same information as before (horizontal bar CFR) but now with dot-plot style
# and both metrics on one panel for direct comparison
cfr_df = agg_df.sort_values('fatality_rate_total', ascending=True).copy()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=cfr_df['fatality_rate_total'], y=cfr_df['Region'],
    mode='markers+text',
    name='Cumulative CFR',
    text=[f'{v:.2f}%' for v in cfr_df['fatality_rate_total']],
    textposition='middle right',
    textfont=dict(size=10, color='#1d4ed8'),
    marker=dict(color='#2563eb', size=14, line=dict(color='white', width=1.5)),
    hovertemplate='<b>%{y}</b><br>Cumulative CFR: %{x:.2f}%<extra></extra>'
))
fig.add_trace(go.Scatter(
    x=cfr_df['fatality_rate_24h'], y=cfr_df['Region'],
    mode='markers',
    name='24-Hour CFR',
    marker=dict(color='#f97316', size=10, symbol='diamond',
                line=dict(color='white', width=1)),
    hovertemplate='<b>%{y}</b><br>24h CFR: %{x:.2f}%<extra></extra>'
))
# Connector lines for readability
for _, row in cfr_df.iterrows():
    fig.add_shape(type='line',
        x0=0, x1=row['fatality_rate_total'], y0=row['Region'], y1=row['Region'],
        line=dict(color='#e2e8f0', width=2))

fig.update_layout(
    **LIGHT,
    title='Case-Fatality Rate by WHO Region — Overall vs 24-Hour Snapshot',
    xaxis_title='Case-Fatality Rate (%)',
    height=360,
    showlegend=True,
)
apply_light_axes(fig)
fig.update_xaxes(ticksuffix='%')
fig.show()

In [ ]:
# ── Bubble chart: cases vs deaths, sized by country count (same as before)
fig = px.scatter(
    agg_df,
    x='cases_total', y='deaths_total',
    size='n_countries', color='Region',
    text='Region',
    color_discrete_sequence=PALETTE,
    labels={'cases_total': 'Cumulative Cases', 'deaths_total': 'Cumulative Deaths',
            'n_countries': 'Countries in Region'},
    size_max=55,
    title='Cases vs Deaths by Region — Bubble Size Represents Country Count',
)
fig.update_traces(
    textposition='top center',
    textfont=dict(color='#374151', size=10),
    marker=dict(opacity=0.75, line=dict(color='white', width=1.5))
)
fig.update_layout(
    **LIGHT,
    xaxis_title='Cumulative Cases',
    yaxis_title='Cumulative Deaths',
    height=450,
)
apply_light_axes(fig)
fig.show()

In [ ]:
# ── NEW: Treemap of cases by region (shows hierarchical scale clearly) ────────
fig = px.treemap(
    agg_df,
    path=['Region'],
    values='cases_total',
    color='fatality_rate_total',
    color_continuous_scale='RdYlGn_r',
    title='Case Volume by Region — Colour Encodes Case-Fatality Rate',
    labels={'cases_total': 'Cumulative Cases', 'fatality_rate_total': 'CFR (%)'},
    hover_data={'cases_total': ':,.0f', 'fatality_rate_total': ':.2f'},
)
fig.update_traces(
    textfont_size=13,
    textfont_color='white',
    hovertemplate='<b>%{label}</b><br>Cases: %{value:,.0f}<br>CFR: %{color:.2f}%<extra></extra>'
)
fig.update_layout(
    **LIGHT,
    coloraxis_colorbar=dict(title='CFR (%)', tickfont=dict(size=11, color='#374151')),
    height=360,
)
fig.show()

In [ ]:
# ── Comparative Case Study: USA vs India ──────────────────────────────
comp_df = df_table[df_table['Name'].isin(['United States of America', 'India'])].copy()
comp_df['CFR'] = 100 * comp_df['Deaths - cumulative total'] / comp_df['Cases - cumulative total']

fig = make_subplots(rows=1, cols=3, 
                    subplot_titles=['Cumulative Cases', 'Cumulative Deaths', 'Case Fatality Rate (%)'],
                    horizontal_spacing=0.1)

colors = ['#1d4ed8', '#15803d'] # USA blue, India green

fig.add_trace(go.Bar(x=comp_df['Name'], y=comp_df['Cases - cumulative total'], marker_color=colors, showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(x=comp_df['Name'], y=comp_df['Deaths - cumulative total'], marker_color=colors, showlegend=False), row=1, col=2)
fig.add_trace(go.Bar(x=comp_df['Name'], y=comp_df['CFR'], marker_color=colors, showlegend=False, text=comp_df['CFR'].round(2).astype(str)+'%', textposition='auto'), row=1, col=3)

fig.update_layout(
    **LIGHT,
    title='Direct Comparison: United States vs. India (Scale vs. Fatality)',
    height=450
)
apply_light_axes(fig)
fig.show()

<div style="background: #fefce8; border: 1px solid #fde68a; border-radius: 6px; padding: 16px 22px; margin: 8px 0 24px;">
  <p style="color: #7c2d12; font-family: 'Inter', sans-serif; line-height: 1.75; margin: 0; font-size: 0.94em;">
    <strong>Finding:</strong> While the United States registered substantially higher cumulative deaths and a higher explicit case-fatality rate, India experienced an intensely concentrated wave of cases. The disparity in aggregate fatality rates reflects complex factors such as demographic age structures, reporting methodologies, and differing healthcare infrastructure strain during variant surges.
  </p>
</div>

<div style="border-top: 2px solid #e2e8f0; padding-top: 32px; margin: 36px 0 20px;">
  <span style="background: #fef9ec; color: #a16207; font-size: 0.75em; font-weight: 600; font-family: 'Inter', sans-serif; text-transform: uppercase; letter-spacing: 0.08em; padding: 4px 10px; border-radius: 4px;">Section 3</span>
  <h2 id="section-3" style="color: #0f172a; font-family: 'Inter', sans-serif; font-size: 1.35em; font-weight: 700; margin: 10px 0 4px;">Country-Level Wave Patterns</h2>
  <p style="color: #64748b; font-family: 'Inter', sans-serif; font-size: 0.95em; margin: 0;">Timeline evolution of cases and deaths for the five most-affected countries.</p>
</div>

In [ ]:
# ── Load time-series data ────────────────────────────────────────────────────
df_ts = pd.read_csv('./data/WHO-COVID-19-global-data.csv')
df_ts['Date_reported'] = pd.to_datetime(df_ts['Date_reported'])

covid_rank = covid_df.sort_values('Cases - cumulative total', ascending=False)
top5 = list(covid_rank.iloc[1:6]['Name'])
print('Top 5 countries:', top5)

wave_colors = ['#2563eb', '#dc2626', '#16a34a', '#d97706', '#7c3aed']

In [ ]:
# ── Overlaid 7-day rolling new cases (same data as before, now single panel for comparison)
# Previously shown as 5 separate stacked subplots; now one overlay for direct comparison
fig = go.Figure()

for i, country in enumerate(top5):
    sub = df_ts[df_ts['Country'] == country].sort_values('Date_reported').copy()
    sub['rolling'] = sub['New_cases'].clip(lower=0).rolling(7, min_periods=1).mean()

    fig.add_trace(go.Scatter(
        x=sub['Date_reported'], y=sub['rolling'],
        name=country, mode='lines',
        line=dict(color=wave_colors[i], width=1.8),
        hovertemplate=f'<b>{country}</b><br>%{{x|%b %Y}}: %{{y:,.0f}} (7d avg)<extra></extra>'
    ))

fig.update_layout(
    **LIGHT,
    title='Daily New Cases — 7-Day Rolling Average, Top 5 Countries',
    xaxis_title='Date',
    yaxis_title='New Cases (7-day average)',
    height=420,
)
apply_light_axes(fig)
fig.show()

In [ ]:
# ── Cumulative cases — log scale (same as before, light styling) ─────────────
fig = go.Figure()

for i, country in enumerate(top5):
    sub = df_ts[df_ts['Country'] == country].sort_values('Date_reported').copy()
    sub['cumulative'] = sub['New_cases'].clip(lower=0).cumsum()

    fig.add_trace(go.Scatter(
        x=sub['Date_reported'], y=sub['cumulative'],
        name=country, mode='lines',
        line=dict(color=wave_colors[i], width=2),
        hovertemplate=f'<b>{country}</b><br>%{{x|%b %Y}}: %{{y:,.0f}}<extra></extra>'
    ))

fig.update_layout(
    **LIGHT,
    title='Cumulative Cases (Log Scale) — Top 5 Countries',
    xaxis_title='Date',
    yaxis=dict(title='Cumulative Cases', type='log'),
    height=400,
)
apply_light_axes(fig)
fig.show()

In [ ]:
# ── Monthly heatmap (same data, enhanced light styling) ──────────────────────
df_top5 = df_ts[df_ts['Country'].isin(top5)].copy()
df_top5['YearMonth'] = df_top5['Date_reported'].dt.to_period('M').astype(str)

hmap = df_top5.groupby(['Country', 'YearMonth'])['New_cases'].sum().reset_index()
hmap_pivot = hmap.pivot(index='Country', columns='YearMonth', values='New_cases').fillna(0)

fig = go.Figure(go.Heatmap(
    z=hmap_pivot.values,
    x=list(hmap_pivot.columns),
    y=list(hmap_pivot.index),
    colorscale='Blues',
    hovertemplate='<b>%{y}</b><br>%{x}: %{z:,.0f} cases<extra></extra>',
    colorbar=dict(title='Cases', tickfont=dict(color='#374151', size=10))
))

fig.update_layout(
    **LIGHT,
    title='Monthly Case Volume — Wave Patterns by Country',
    xaxis=dict(title='Month', tickangle=-45, tickfont=dict(size=9, color='#6b7280')),
    yaxis=dict(title=''),
    height=340,
)
fig.show()

In [ ]:
# ── NEW: Top 20 countries by cumulative cases (ranking bar) ──────────────────
top20 = covid_rank[covid_rank['Name'] != 'Global'].iloc[:20].copy()
top20 = top20.sort_values('Cases - cumulative total', ascending=True)

# Colour by region
region_color_map = dict(zip(regions, PALETTE))
bar_colors = [region_color_map.get(r, '#94a3b8') for r in top20['WHO Region']]

fig = go.Figure(go.Bar(
    x=top20['Cases - cumulative total'],
    y=top20['Name'],
    orientation='h',
    marker_color=bar_colors,
    marker_line_width=0,
    hovertemplate='<b>%{y}</b><br>Cases: %{x:,.0f}<extra></extra>'
))

fig.update_layout(
    **LIGHT,
    title='Top 20 Countries by Cumulative COVID-19 Cases',
    xaxis_title='Cumulative Cases',
    yaxis_title='',
    height=540,
)
apply_light_axes(fig)
fig.show()

In [ ]:
# -- Stacked area chart: cumulative deaths, top 5 countries
# Shows cumulative death burden accumulation over time
import re

def hex_to_rgba(hex_color, alpha=0.18):
    hex_color = hex_color.lstrip('#')
    r, g, b = int(hex_color[0:2],16), int(hex_color[2:4],16), int(hex_color[4:6],16)
    return f'rgba({r},{g},{b},{alpha})'

fig = go.Figure()

for i, country in enumerate(top5):
    sub = df_ts[df_ts['Country'] == country].sort_values('Date_reported').copy()
    sub['cum_deaths'] = sub['New_deaths'].clip(lower=0).cumsum()

    fig.add_trace(go.Scatter(
        x=sub['Date_reported'], y=sub['cum_deaths'],
        name=country, mode='lines',
        fill='tonexty' if i > 0 else 'tozeroy',
        line=dict(color=wave_colors[i], width=1.5),
        fillcolor=hex_to_rgba(wave_colors[i], 0.15),
        hovertemplate=f'<b>{country}</b><br>%{{x|%b %Y}}: %{{y:,.0f}} deaths<extra></extra>'
    ))

fig.update_layout(
    **LIGHT,
    title='Cumulative Deaths — Stacked Area, Top 5 Countries',
    xaxis_title='Date',
    yaxis_title='Cumulative Deaths',
    height=400,
)
apply_light_axes(fig)
fig.show()

<div style="background: #fffbeb; border: 1px solid #fde68a; border-radius: 6px; padding: 16px 22px; margin: 8px 0 24px;">
  <p style="color: #7c2d12; font-family: 'Inter', sans-serif; line-height: 1.75; margin: 0; font-size: 0.94em;">
    <strong>Wave analysis:</strong> India shows three distinct peaks (Sep 2020, May 2021, Jan 2022), with the second wave being the deadliest. The USA and Brazil show broader and larger surges. Germany and France saw their biggest surges only in late 2021&ndash;2022 with the Omicron variant.
  </p>
</div>

<div style="border-top: 2px solid #e2e8f0; padding-top: 32px; margin: 36px 0 20px;">
  <span style="background: #fef2f2; color: #991b1b; font-size: 0.75em; font-weight: 600; font-family: 'Inter', sans-serif; text-transform: uppercase; letter-spacing: 0.08em; padding: 4px 10px; border-radius: 4px;">Section 4</span>
  <h2 id="section-4" style="color: #0f172a; font-family: 'Inter', sans-serif; font-size: 1.35em; font-weight: 700; margin: 10px 0 4px;">Global Geographic View</h2>
  <p style="color: #64748b; font-family: 'Inter', sans-serif; font-size: 0.95em; margin: 0;">Snapshot of the pandemic's geographic footprint as of January 31, 2022.</p>
</div>

In [ ]:
# ── Load OWID dataset ────────────────────────────────────────────────────────
owid = pd.read_csv('./data/owid-covid-data.csv')
owid_clean = owid.dropna(subset=['iso_code', 'location', 'continent', 'date', 'total_cases'])
owid_clean = owid_clean.sort_values('date')

snap_date = '2022-01-31'
snap = owid_clean[owid_clean['date'] == snap_date].copy()
snap['total_deaths'] = snap['total_deaths'].fillna(0)
snap['log_cases'] = np.log1p(snap['total_cases'])
print(f'Snapshot: {snap_date} | {len(snap)} countries')

In [ ]:
# ── Choropleth — total cases (same, light map style) ─────────────────────────
fig = px.choropleth(
    snap,
    locations='iso_code',
    color='log_cases',
    hover_name='location',
    hover_data={'total_cases': ':,.0f', 'total_deaths': ':,.0f', 'log_cases': False},
    color_continuous_scale='Blues',
    projection='natural earth',
    title=f'Total Confirmed Cases per Country — {snap_date} (log-scale colour)',
    labels={'log_cases': 'log(cases+1)', 'total_cases': 'Cases', 'total_deaths': 'Deaths'}
)
fig.update_layout(
    **LIGHT,
    geo=dict(
        bgcolor='#f8fafc',
        landcolor='#e2e8f0',
        oceancolor='#f0f4f8',
        showocean=True, showland=True, showcoastlines=True,
        coastlinecolor='#cbd5e1',
        lakecolor='#f0f4f8',
        framecolor='#e2e8f0'
    ),
    coloraxis_colorbar=dict(title='log(cases)', tickfont=dict(size=10, color='#374151')),
    height=480,
)
fig.show()

In [ ]:
# ── Bubble map by continent (same data, light earth style) ───────────────────
fig = px.scatter_geo(
    snap,
    locations='iso_code',
    color='continent',
    hover_name='location',
    size='total_cases',
    projection='natural earth',
    title=f'Case Bubble Map by Continent — {snap_date}',
    color_discrete_sequence=PALETTE8,
    size_max=45,
    labels={'total_cases': 'Total Cases'}
)
fig.update_traces(marker=dict(opacity=0.65, line=dict(color='white', width=0.5)))
fig.update_layout(
    **LIGHT,
    geo=dict(
        bgcolor='#f8fafc',
        landcolor='#e2e8f0',
        oceancolor='#f0f4f8',
        showocean=True, showland=True, showcoastlines=True,
        coastlinecolor='#cbd5e1',
        framecolor='#e2e8f0'
    ),
    height=480,
)
fig.show()

<div style="border-top: 2px solid #e2e8f0; padding-top: 32px; margin: 36px 0 20px;">
  <span style="background: #f0f9ff; color: #075985; font-size: 0.75em; font-weight: 600; font-family: 'Inter', sans-serif; text-transform: uppercase; letter-spacing: 0.08em; padding: 4px 10px; border-radius: 4px;">Section 5</span>
  <h2 id="section-5" style="color: #0f172a; font-family: 'Inter', sans-serif; font-size: 1.35em; font-weight: 700; margin: 10px 0 4px;">Deaths and Vaccinations</h2>
  <p style="color: #64748b; font-family: 'Inter', sans-serif; font-size: 0.95em; margin: 0;">Did vaccination uptake correlate with reduced death rates by early 2022?</p>
</div>

In [ ]:
# ── Scatter: vaccinations vs total deaths, sized by cases (same as before) ───
vax_snap = owid_clean[owid_clean['date'] == snap_date].copy()
vax_snap = vax_snap.dropna(subset=['total_vaccinations_per_hundred', 'total_deaths'])
vax_snap = vax_snap[vax_snap['continent'].notna()]

fig = px.scatter(
    vax_snap,
    x='total_vaccinations_per_hundred',
    y='total_deaths',
    color='continent',
    size='total_cases',
    hover_name='location',
    log_y=True,
    trendline='ols',
    title='Vaccinations per 100 People vs Total Deaths (log scale) — January 2022',
    labels={
        'total_vaccinations_per_hundred': 'Vaccinations per 100 People',
        'total_deaths': 'Total Deaths (log scale)',
        'total_cases': 'Total Cases'
    },
    color_discrete_sequence=PALETTE8,
    size_max=38,
)
fig.update_traces(marker=dict(opacity=0.7, line=dict(color='white', width=0.5)),
                  selector=dict(mode='markers'))
fig.update_layout(**LIGHT, height=460)
apply_light_axes(fig)
fig.show()

In [ ]:
# ── NEW: Box plot — deaths per million by continent (distribution view) ───────
box_df = owid_clean[owid_clean['date'] == snap_date].copy()
box_df = box_df.dropna(subset=['total_deaths_per_million', 'continent'])
box_df = box_df[box_df['continent'] != '']

fig = px.box(
    box_df.sort_values('continent'),
    x='continent',
    y='total_deaths_per_million',
    color='continent',
    points='outliers',
    title='Deaths per Million — Distribution Across Countries by Continent',
    labels={
        'continent': 'Continent',
        'total_deaths_per_million': 'Deaths per Million'
    },
    color_discrete_sequence=PALETTE8,
)
fig.update_traces(marker=dict(opacity=0.65, size=4))
fig.update_layout(**LIGHT, height=420, showlegend=False)
apply_light_axes(fig)
fig.show()

<div style="background: #f0f9ff; border: 1px solid #bae6fd; border-radius: 6px; padding: 16px 22px; margin: 8px 0 24px;">
  <p style="color: #0c4a6e; font-family: 'Inter', sans-serif; line-height: 1.75; margin: 0; font-size: 0.94em;">
    <strong>Vaccination note:</strong> By January 2022 higher vaccination rates did not always translate to lower raw death counts &mdash; wealthier nations with higher uptake also had larger older populations and earlier waves. The relationship is complex, but vaccination clearly buffered the severity of the Omicron wave relative to earlier variants.
  </p>
</div>

<div style="border-top: 3px solid #0f172a; padding-top: 32px; margin: 48px 0 8px;">
  <h2 id="conclusions" style="color: #0f172a; font-family: 'Inter', sans-serif; font-size: 1.3em; font-weight: 700; margin: 0 0 18px;">Conclusions</h2>
  <table style="border-collapse: collapse; width: 100%; font-family: 'Inter', sans-serif; font-size: 0.93em;">
    <tr style="background: #f8fafc;">
      <td style="padding: 10px 16px; border: 1px solid #e2e8f0; color: #0f172a; font-weight: 600; width: 220px;">Regional disparity</td>
      <td style="padding: 10px 16px; border: 1px solid #e2e8f0; color: #374151;">Europe and the Americas bear the heaviest cumulative burden despite not being the most populous.</td>
    </tr>
    <tr>
      <td style="padding: 10px 16px; border: 1px solid #e2e8f0; color: #0f172a; font-weight: 600;">Africa</td>
      <td style="padding: 10px 16px; border: 1px solid #e2e8f0; color: #374151;">Low reported numbers likely reflect a mix of demographic advantages and reporting gaps rather than true sparing.</td>
    </tr>
    <tr style="background: #f8fafc;">
      <td style="padding: 10px 16px; border: 1px solid #e2e8f0; color: #0f172a; font-weight: 600;">India's second wave</td>
      <td style="padding: 10px 16px; border: 1px solid #e2e8f0; color: #374151;">The May 2021 peak was the most devastating single-country surge in the dataset, driven by the Delta variant.</td>
    </tr>
    <tr>
      <td style="padding: 10px 16px; border: 1px solid #e2e8f0; color: #0f172a; font-weight: 600;">Case-fatality rates</td>
      <td style="padding: 10px 16px; border: 1px solid #e2e8f0; color: #374151;">Wide variation across regions, driven by healthcare capacity, median age, and dominant variant.</td>
    </tr>
    <tr style="background: #f8fafc;">
      <td style="padding: 10px 16px; border: 1px solid #e2e8f0; color: #0f172a; font-weight: 600;">Vaccination rollout</td>
      <td style="padding: 10px 16px; border: 1px solid #e2e8f0; color: #374151;">By early 2022 most high-income nations had achieved high uptake, but coverage remained deeply uneven globally.</td>
    </tr>
  </table>
</div>